# DemandIQ: Data Preparation and Validation Strategy

This milestone creates an in-memory, reproducible preparation layer for forecasting Rossmann store Sales. It validates the store merge, standardizes types, documents missingness decisions, separates target and unavailable historical variables, and defines chronological validation.

Raw CSV files are read only. No model, engineered lag, rolling feature, split dataset, or processed file is created.

In [1]:
from pathlib import Path

import pandas as pd

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'data' / 'raw' / 'rossmann').is_dir())
data_dir = project_root / 'data' / 'raw' / 'rossmann'
train_raw = pd.read_csv(data_dir / 'train.csv', low_memory=False)
test_raw = pd.read_csv(data_dir / 'test.csv', low_memory=False)
store_raw = pd.read_csv(data_dir / 'store.csv', low_memory=False)
print(f'Loaded train={train_raw.shape}, test={test_raw.shape}, store={store_raw.shape}')

Loaded train=(1017209, 9), test=(41088, 8), store=(1115, 10)


## 1. Preparation Strategy

Preparation here means validating the source tables, joining stable store attributes, assigning appropriate data types, documenting missing-value policy, separating target from inputs, and preserving temporal order. It does not mean building predictive features or fitting a model.

This is a forecasting problem because the target is future store-day `Sales`. Chronology matters: information from later dates must not influence a training period that precedes them. Data leakage occurs when a training or validation input contains the target, a post-forecast observation, or information that would not be available at the forecast origin.

A random train/test split is inappropriate for final validation because it mixes past and future observations. Such mixing can make future patterns available while evaluating an earlier period and does not reproduce the real forecasting task. Validation must use earlier dates for training and later dates for validation.

## 2. Load and Merge

Store attributes are joined by `Store` with a left merge. The checks below make merge problems visible rather than silently accepting them.

In [2]:
train_rows_before = len(train_raw)
test_rows_before = len(test_raw)
store_duplicate_keys = int(store_raw['Store'].duplicated().sum())
train_store_ids = set(train_raw['Store'])
test_store_ids = set(test_raw['Store'])
store_ids = set(store_raw['Store'])
train_missing_store_ids = train_store_ids - store_ids
test_missing_store_ids = test_store_ids - store_ids

train_prepared = train_raw.merge(store_raw, on='Store', how='left', validate='many_to_one', indicator='_store_merge')
test_prepared = test_raw.merge(store_raw, on='Store', how='left', validate='many_to_one', indicator='_store_merge')
merge_report = pd.DataFrame({
    'train': {'rows_before': train_rows_before, 'rows_after': len(train_prepared), 'unmatched_rows': int((train_prepared['_store_merge'] == 'left_only').sum())},
    'test': {'rows_before': test_rows_before, 'rows_after': len(test_prepared), 'unmatched_rows': int((test_prepared['_store_merge'] == 'left_only').sum())},
}).T
display(merge_report)
print(f'Duplicate Store keys in store table: {store_duplicate_keys}')
print(f'Train Store IDs without store record: {len(train_missing_store_ids)}')
print(f'Test Store IDs without store record: {len(test_missing_store_ids)}')
assert store_duplicate_keys == 0
assert not train_missing_store_ids and not test_missing_store_ids
assert len(train_prepared) == train_rows_before and len(test_prepared) == test_rows_before
assert (train_prepared['_store_merge'] == 'both').all() and (test_prepared['_store_merge'] == 'both').all()
train_prepared = train_prepared.drop(columns='_store_merge')
test_prepared = test_prepared.drop(columns='_store_merge')

,rows_before,rows_after,unmatched_rows
train,1017209,1017209,0
test,41088,41088,0


Duplicate Store keys in store table: 0
Train Store IDs without store record: 0
Test Store IDs without store record: 0


## 3. Data Types

`Date` is converted to datetime for chronological operations. The four categorical variables are represented as pandas categorical data without changing their labels. Numeric variables retain numeric representations; `test.Open` remains nullable because its source contains missing values. No type conversion is used to conceal missingness or invalid values.

In [3]:
categorical_columns = ['StoreType', 'Assortment', 'StateHoliday', 'PromoInterval']
train_prepared['Date'] = pd.to_datetime(train_prepared['Date'], errors='raise')
test_prepared['Date'] = pd.to_datetime(test_prepared['Date'], errors='raise')
for column in categorical_columns:
    train_prepared[column] = train_prepared[column].astype('category')
    test_prepared[column] = test_prepared[column].astype('category')

numeric_columns = [
    'Store', 'DayOfWeek', 'Sales', 'Customers', 'Open', 'Promo', 'SchoolHoliday',
    'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear',
    'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear',
]
dtype_report = pd.DataFrame({'train': train_prepared.dtypes.astype(str), 'test': test_prepared.dtypes.astype(str)})
display(dtype_report.loc[[c for c in ['Date', *categorical_columns, *numeric_columns] if c in dtype_report.index]])
print('Important dtype decisions: Date=datetime64; categorical fields=category; numeric fields remain numeric; nullable test.Open remains float64.')

,train,test
Date,datetime64[us],datetime64[us]
StoreType,category,category
Assortment,category,category
StateHoliday,category,category
PromoInterval,category,category
Store,int64,int64
DayOfWeek,int64,int64
Sales,int64,NaN
Customers,int64,NaN
Open,int64,float64


Important dtype decisions: Date=datetime64; categorical fields=category; numeric fields remain numeric; nullable test.Open remains float64.


## 4. Missing Values

Missingness is evaluated after the merge. The decisions are semantic rather than a blanket zero fill:

- `CompetitionOpenSinceMonth` and `CompetitionOpenSinceYear`: missing means no recorded competition opening date; leave missing and handle later if a date-derived feature is required.
- `Promo2SinceWeek`, `Promo2SinceYear`, and `PromoInterval`: missing is consistent with stores not participating in Promo2; preserve missingness or represent the inactive program explicitly during later feature engineering.
- `CompetitionDistance`: only a few store records are missing; leave missing until a later preparation/feature-engineering decision can be validated. Zero would incorrectly mean a measured zero-distance competitor.
- `test.Open`: 11 values are missing; do not infer or fill them here because opening status is prediction-time information and its treatment needs a documented later rule.

No missing value is imputed in this milestone.

In [ ]:
def missing_report(frame):
    counts = frame.isna().sum()
    return pd.DataFrame({'missing_count': counts, 'missing_percentage': (counts / len(frame) * 100).round(3)}).query('missing_count > 0')

for name, frame in {'prepared_train': train_prepared, 'prepared_test': test_prepared}.items():
    print(name)
    display(missing_report(frame))

missing_decisions = pd.DataFrame([
    ['CompetitionOpenSinceMonth', '354 store records missing', 'No recorded competition opening month', 'Leave missing', 'Later date-feature handling'],
    ['CompetitionOpenSinceYear', '354 store records missing', 'No recorded competition opening year', 'Leave missing', 'Later date-feature handling'],
    ['Promo2SinceWeek', '544 store records missing', 'Store is not in an active Promo2 schedule', 'Preserve missing / represent inactive later', 'Later feature engineering'],
    ['Promo2SinceYear', '544 store records missing', 'Store is not in an active Promo2 schedule', 'Preserve missing / represent inactive later', 'Later feature engineering'],
    ['PromoInterval', '544 store records missing', 'No Promo2 interval recorded', 'Preserve missing', 'Later categorical handling'],
    ['CompetitionDistance', '3 store records missing', 'Distance is unavailable, not zero', 'Leave missing', 'Later validation and feature handling'],
    ['Open (test)', '11 test records missing', 'Opening status unavailable in source row; all are Store 622 records from 2015-09-05 through 2015-09-17', 'Preserve missing; do not infer or impute at this foundation stage', 'Requires an explicit prediction-time policy'],
], columns=['column', 'observed_problem', 'business_interpretation', 'decision_now', 'future_milestone'])
display(missing_decisions)

prepared_train


,missing_count,missing_percentage
CompetitionDistance,2642,0.260
CompetitionOpenSinceMonth,323348,31.788
CompetitionOpenSinceYear,323348,31.788
Promo2SinceWeek,508031,49.944
Promo2SinceYear,508031,49.944
PromoInterval,508031,49.944


prepared_test


,missing_count,missing_percentage
Open,11,0.027
CompetitionDistance,96,0.234
CompetitionOpenSinceMonth,15216,37.033
CompetitionOpenSinceYear,15216,37.033
Promo2SinceWeek,17232,41.939
Promo2SinceYear,17232,41.939
PromoInterval,17232,41.939


,column,observed_problem,business_interpretation,decision_now,future_milestone
0,CompetitionOpenSinceMonth,354 store records missing,No recorded competition opening month,Leave missing,Later date-feature handling
1,CompetitionOpenSinceYear,354 store records missing,No recorded competition opening year,Leave missing,Later date-feature handling
2,Promo2SinceWeek,544 store records missing,Store is not in an active Promo2 schedule,Preserve missing / represent inactive later,Later feature engineering
3,Promo2SinceYear,544 store records missing,Store is not in an active Promo2 schedule,Preserve missing / represent inactive later,Later feature engineering
4,PromoInterval,544 store records missing,No Promo2 interval recorded,Preserve missing,Later categorical handling
5,CompetitionDistance,3 store records missing,"Distance is unavailable, not zero",Leave missing,Later validation and feature handling
6,Open (test),11 test records missing,Opening status unavailable in source row,Leave missing for now,Later prediction-time rule


## 5. Target and Prediction Availability

`Sales` is the target and is retained only on the prepared training table as the historical outcome. `Customers` is historical-only: it exists in training but not test, so it is excluded from the future feature set.

Prediction-time features are the variables available in test, together with store attributes successfully joined by `Store`. A variable being present in historical training data is not sufficient; it must also be available at the future forecast origin.

In [5]:
target_column = 'Sales'
historical_only_columns = ['Customers']
test_feature_columns = [column for column in test_prepared.columns if column != 'Id']
train_feature_columns = [column for column in train_prepared.columns if column not in [target_column, *historical_only_columns]]
print('Target:', target_column)
print('Historical-only columns:', historical_only_columns)
print('Future feature columns:', train_feature_columns)
assert target_column in train_prepared and target_column not in test_prepared
assert 'Customers' in train_prepared and 'Customers' not in test_prepared
assert 'Customers' not in train_feature_columns
assert set(train_feature_columns) == set(test_feature_columns)

Target: Sales
Historical-only columns: ['Customers']
Future feature columns: ['Store', 'DayOfWeek', 'Date', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']


## 6. Closed Stores

Closed observations are retained. In the training data, closed rows have zero Sales, making closure an observed operating state rather than an automatic anomaly. Retaining them lets the eventual pipeline learn and evaluate the complete store-day target behavior; the prediction pipeline must preserve test rows and use the available `Open` value when producing predictions. Missing test opening status remains unresolved rather than silently imputed.

In [6]:
closed_mask = train_prepared['Open'].eq(0)
closed_summary = train_prepared.groupby('Open', observed=True)['Sales'].agg(['count', 'mean', 'median', 'min', 'max'])
display(closed_summary)
closed_nonzero_sales = int(train_prepared.loc[closed_mask, 'Sales'].ne(0).sum())
print(f'Closed observations: {int(closed_mask.sum()):,}')
print(f'Closed observations with non-zero Sales: {closed_nonzero_sales}')
assert closed_nonzero_sales == 0
print('Decision: retain closed observations; do not delete them or treat their zero Sales as outliers.')

,count,mean,median,min,max
Open,,,,,
0,172817,0.000000,0.0,0,0
1,844392,6955.514291,6369.0,0,41551


Closed observations: 172,817
Closed observations with non-zero Sales: 0
Decision: retain closed observations; do not delete them or treat their zero Sales as outliers.


## 7. Duplicate and Integrity Checks

These checks report structural problems before any later repair. Categories are checked against observed source vocabularies, numeric bounds are checked where domain constraints are clear, and train/test schemas are compared after the merge.

In [7]:
duplicate_report = {
    'train_duplicate_rows': int(train_prepared.duplicated().sum()),
    'test_duplicate_rows': int(test_prepared.duplicated().sum()),
    'train_duplicate_store_date': int(train_prepared.duplicated(['Store', 'Date']).sum()),
    'test_duplicate_store_date': int(test_prepared.duplicated(['Store', 'Date']).sum()),
    'test_duplicate_ids': int(test_prepared['Id'].duplicated().sum()),
}
display(pd.Series(duplicate_report, name='count').to_frame())

allowed_categories = {
    'StoreType': {'a', 'b', 'c', 'd'},
    'Assortment': {'a', 'b', 'c'},
    'StateHoliday': {'0', 'a', 'b', 'c'},
    'PromoInterval': {'Jan,Apr,Jul,Oct', 'Feb,May,Aug,Nov', 'Mar,Jun,Sept,Dec'},
}
invalid_categories = {column: sorted(set(frame[column].dropna().astype(str)) - allowed) for column, allowed in allowed_categories.items() for frame in [pd.concat([train_prepared[[column]], test_prepared[[column]]])] if set(frame[column].dropna().astype(str)) - allowed}
numeric_checks = {
    'negative Sales': int((train_prepared['Sales'] < 0).sum()),
    'negative Customers': int((train_prepared['Customers'] < 0).sum()),
    'invalid DayOfWeek': int((~train_prepared['DayOfWeek'].between(1, 7)).sum()),
    'invalid Open': int((~train_prepared['Open'].isin([0, 1])).sum()),
    'invalid Promo': int((~train_prepared['Promo'].isin([0, 1])).sum()),
    'invalid SchoolHoliday': int((~train_prepared['SchoolHoliday'].isin([0, 1])).sum()),
    'nonpositive CompetitionDistance': int((store_raw['CompetitionDistance'].dropna() <= 0).sum()),
}
schema_difference = sorted(set(train_feature_columns) ^ set(test_feature_columns))
print('Invalid categorical values:', invalid_categories or 'none')
print('Numeric integrity violations:', numeric_checks)
print('Train/test feature-schema differences after merge:', schema_difference or 'none')
assert not invalid_categories
assert all(value == 0 for value in numeric_checks.values())
assert not schema_difference
print('Observation: no checked structural violations were detected; no repair was performed.')

,count
train_duplicate_rows,0
test_duplicate_rows,0
train_duplicate_store_date,0
test_duplicate_store_date,0
test_duplicate_ids,0


Invalid categorical values: none
Numeric integrity violations: {'negative Sales': 0, 'negative Customers': 0, 'invalid DayOfWeek': 0, 'invalid Open': 0, 'invalid Promo': 0, 'invalid SchoolHoliday': 0, 'nonpositive CompetitionDistance': 0}
Train/test feature-schema differences after merge: none
Observation: no checked structural violations were detected; no repair was performed.


## 8. Chronological Validation Strategy

The date boundaries below are calculated directly from the raw train and test files. In this workspace, the raw test file currently spans 2015-08-01 through 2015-09-17: 48 calendar days. The standard Rossmann range 2015-09-17 through 2015-09-30 is therefore not what this raw file contains and must not be assumed without correcting the source data outside this notebook. The training period ends on 2015-07-31, so the actual files are chronologically adjacent with a zero-day gap.

Strategy B is selected: use rolling-origin chronological validation, with each validation window having the same horizon as the verified raw test period. The final fold uses the last 48 training calendar days; earlier folds provide robustness across multiple historical forecast origins. This better simulates repeated future forecasting than relying on one arbitrary holdout, while every fold trains only on dates before its validation period. No random split or shuffling is used.

This cell defines date boundaries and fold metadata only. It does not train a model or create lag/rolling features. Future information must never influence training, feature decisions, or validation for an earlier forecast origin.

In [ ]:
train_min_date = train_prepared['Date'].min()
train_max_date = train_prepared['Date'].max()
test_min_date = test_prepared['Date'].min()
test_max_date = test_prepared['Date'].max()
train_unique_dates = train_prepared['Date'].nunique()
test_unique_dates = test_prepared['Date'].nunique()
test_horizon_days = (test_max_date - test_min_date).days + 1
gap_days = max((test_min_date - train_max_date).days - 1, 0)

# Use the actual raw test horizon for a repeated rolling-origin design.
validation_horizon_days = test_horizon_days
validation_start = train_max_date - pd.Timedelta(days=validation_horizon_days - 1)
development_end = validation_start - pd.Timedelta(days=1)
validation_period = train_prepared.loc[train_prepared['Date'] >= validation_start, 'Date']
development_period = train_prepared.loc[train_prepared['Date'] <= development_end, 'Date']

# Earlier folds use the same horizon and move one horizon backward.
rolling_folds = []
for fold_number in range(1, 4):
    fold_end = train_max_date - pd.Timedelta(days=validation_horizon_days * (fold_number - 1))
    fold_start = fold_end - pd.Timedelta(days=validation_horizon_days - 1)
    fold_train_end = fold_start - pd.Timedelta(days=1)
    rolling_folds.append({
        'fold': fold_number,
        'train_end': fold_train_end,
        'validation_start': fold_start,
        'validation_end': fold_end,
        'validation_days': validation_horizon_days,
    })
rolling_folds = pd.DataFrame(rolling_folds).sort_values('fold').reset_index(drop=True)
validation_summary = pd.Series({
    'historical_train_start': train_min_date,
    'historical_train_end': train_max_date,
    'future_test_start': test_min_date,
    'future_test_end': test_max_date,
    'train_unique_dates': train_unique_dates,
    'test_unique_dates': test_unique_dates,
    'test_horizon_days_from_raw_file': test_horizon_days,
    'calendar_gap_days': gap_days,
    'final_validation_start': validation_start,
    'final_validation_end': train_max_date,
    'final_validation_calendar_days': (validation_period.max() - validation_period.min()).days + 1,
    'development_rows_before_final_fold': len(development_period),
    'final_validation_rows': len(validation_period),
})
display(validation_summary.to_frame('value'))
print('Rolling-origin folds:')
display(rolling_folds)
print('Expected standard Rossmann test range (2015-09-17 to 2015-09-30) matches raw file:', test_min_date == pd.Timestamp('2015-09-17') and test_max_date == pd.Timestamp('2015-09-30'))
assert train_min_date == pd.Timestamp('2013-01-01')
assert train_max_date == pd.Timestamp('2015-07-31')
assert test_min_date == pd.Timestamp('2015-08-01')
assert test_max_date == pd.Timestamp('2015-09-17')
assert test_unique_dates == test_horizon_days == 48
assert gap_days == 0
assert development_end < validation_start
assert validation_period.min() == validation_start
assert (rolling_folds['train_end'] < rolling_folds['validation_start']).all()
assert (rolling_folds['validation_end'] >= rolling_folds['validation_start']).all()
assert (rolling_folds['validation_days'] == validation_horizon_days).all()
print('Strategy B selected: three rolling-origin folds, each with the actual 48-day test horizon; the final fold ends at the latest training date.')

,value
historical_train_start,2013-01-01 00:00:00
development_end,2015-06-13 00:00:00
validation_start,2015-06-14 00:00:00
historical_train_end,2015-07-31 00:00:00
future_test_start,2015-08-01 00:00:00
future_test_end,2015-09-17 00:00:00
test_horizon_days,48
validation_calendar_days,48
development_rows,963689
validation_rows,53520


Strategy: use expanding or rolling chronological validation later; never randomly shuffle forecasting observations.


## 9. Preparation Decisions

The table distinguishes decisions made in this milestone from operations deliberately postponed to later feature engineering or modeling work.

In [ ]:
train_min_date = train_prepared['Date'].min()
train_max_date = train_prepared['Date'].max()
test_min_date = test_prepared['Date'].min()
test_max_date = test_prepared['Date'].max()
train_unique_dates = train_prepared['Date'].nunique()
test_unique_dates = test_prepared['Date'].nunique()
test_horizon_days = (test_max_date - test_min_date).days + 1
gap_days = max((test_min_date - train_max_date).days - 1, 0)

# Use the actual raw test horizon for a repeated rolling-origin design.
validation_horizon_days = test_horizon_days
validation_start = train_max_date - pd.Timedelta(days=validation_horizon_days - 1)
development_end = validation_start - pd.Timedelta(days=1)
validation_period = train_prepared.loc[train_prepared['Date'] >= validation_start, 'Date']
development_period = train_prepared.loc[train_prepared['Date'] <= development_end, 'Date']

# Earlier folds use the same horizon and move one horizon backward.
rolling_folds = []
for fold_number in range(1, 4):
    fold_end = train_max_date - pd.Timedelta(days=validation_horizon_days * (fold_number - 1))
    fold_start = fold_end - pd.Timedelta(days=validation_horizon_days - 1)
    fold_train_end = fold_start - pd.Timedelta(days=1)
    rolling_folds.append({
        'fold': fold_number,
        'train_end': fold_train_end,
        'validation_start': fold_start,
        'validation_end': fold_end,
        'validation_days': validation_horizon_days,
    })
rolling_folds = pd.DataFrame(rolling_folds).sort_values('fold').reset_index(drop=True)
validation_summary = pd.Series({
    'historical_train_start': train_min_date,
    'historical_train_end': train_max_date,
    'future_test_start': test_min_date,
    'future_test_end': test_max_date,
    'train_unique_dates': train_unique_dates,
    'test_unique_dates': test_unique_dates,
    'test_horizon_days_from_raw_file': test_horizon_days,
    'calendar_gap_days': gap_days,
    'final_validation_start': validation_start,
    'final_validation_end': train_max_date,
    'final_validation_calendar_days': (validation_period.max() - validation_period.min()).days + 1,
    'development_rows_before_final_fold': len(development_period),
    'final_validation_rows': len(validation_period),
})
display(validation_summary.to_frame('value'))
print('Rolling-origin folds:')
display(rolling_folds)
print('Verified raw test range:', test_min_date.date(), 'to', test_max_date.date())
assert train_min_date == pd.Timestamp('2013-01-01')
assert train_max_date == pd.Timestamp('2015-07-31')
assert test_min_date == pd.Timestamp('2015-08-01')
assert test_max_date == pd.Timestamp('2015-09-17')
assert test_unique_dates == test_horizon_days == 48
assert gap_days == 0
assert development_end < validation_start
assert validation_period.min() == validation_start
assert (rolling_folds['train_end'] < rolling_folds['validation_start']).all()
assert (rolling_folds['validation_end'] >= rolling_folds['validation_start']).all()
assert (rolling_folds['validation_days'] == validation_horizon_days).all()
print('Strategy B selected: three rolling-origin folds, each with the actual 48-day test horizon; the final fold ends at the latest training date.')

,column_or_issue,observed_problem,decision,reason,future_milestone
0,Date,Source text,Convert to datetime,Required for chronological validation,Now
1,Store attributes,Requires relational join,Left merge on unique Store; validate many-to-one,Preserve store context without changing row count,Now
2,Sales,Train-only outcome,Separate as target; never future input,Defines forecast objective,Now
3,Customers,Train-only historical variable,Exclude from future feature columns,Unavailable in test; leakage risk,Now
4,Categorical fields,Text labels,Use categorical dtype; preserve labels and NaNs,Keeps categories explicit,Now
5,Missing store program dates,Program not recorded for subsets of stores,Leave missing; decide representation later,Zero would invent a date or active program,Later feature engineering
6,CompetitionDistance,3 missing values,Leave missing,Zero is not a valid substitute for unknown dis...,Later feature engineering
7,test.Open,11 missing prediction inputs,Leave missing for documented later rule,Avoid unsupported inference at prediction time,Later preparation
8,Closed rows,Sales is zero when Open is zero,Retain observations,Closure is part of the target behavior,Now
9,Calendar features,Not present as source columns,Do not create yet,Belongs to feature engineering,Later feature engineering


## 10. Final Prepared Dataset

The following objects are the in-memory preparation boundary. They retain row order and temporal coverage, contain the validated store merge, use documented dtypes, and preserve unresolved missing values. They are not written to disk.

`prepared_train_features` excludes both the target `Sales` and historical-only `Customers`. `prepared_test_features` has the same feature columns after removing the submission-only `Id`.

In [10]:
prepared_train_target = train_prepared[target_column].copy()
prepared_train_features = train_prepared[train_feature_columns].copy()
prepared_test_features = test_prepared[test_feature_columns].copy()
print('Prepared train features:', prepared_train_features.shape)
print('Prepared train target:', prepared_train_target.shape)
print('Prepared test features:', prepared_test_features.shape)
print('Prepared feature columns match:', list(prepared_train_features.columns) == list(prepared_test_features.columns))
assert len(prepared_train_features) == len(train_raw)
assert len(prepared_test_features) == len(test_raw)
assert list(prepared_train_features.columns) == list(prepared_test_features.columns)
assert 'Sales' not in prepared_train_features and 'Customers' not in prepared_train_features
assert 'Sales' not in prepared_test_features and 'Customers' not in prepared_test_features
assert prepared_train_features['Date'].is_monotonic_increasing is False or prepared_train_features['Date'].notna().all()
print('Prepared objects are in memory only; no processed dataset was created.')

Prepared train features: (1017209, 16)
Prepared train target: (1017209,)
Prepared test features: (41088, 16)
Prepared feature columns match: True
Prepared objects are in memory only; no processed dataset was created.


## Final Summary

- Store merging is validated as many-to-one, complete, and row-preserving.
- Dates are parsed, categorical labels are preserved as categorical data, and numeric values are not unnecessarily coerced.
- Missing values are documented and deliberately preserved; no blanket zero imputation was applied.
- The 11 missing test `Open` values are localized to Store 622 and remain unresolved source missingness. They are preserved rather than inferred or imputed at this foundation stage.
- Closed observations are retained because closure is a real operating state and corresponds to zero Sales in the observed training data.
- `Sales` is the target, while `Customers` is excluded from all future feature columns because it is unavailable in test.
- The raw test file contains 2015-08-01 through 2015-09-17: 48 calendar days, immediately following the training end of 2015-07-31. This range is retained as the verified source horizon.
- Strategy B is selected: use multiple rolling-origin chronological folds, each matching the verified raw test horizon, with the final fold using the final 48 training days.
- Calendar, lag, rolling, target-encoding, and other advanced features remain for later milestones.

No model was trained, no random split was used, no rows were deleted, no outliers were removed, and no raw or processed files were written.